# web_agent — GOLD dataset run (Kaggle)

Real, leak-safe gold data (before+after images + task). Separate from the synthetic notebook — the synthetic 70k pipeline is untouched. **v14: all 4 pillars trained** — confidence, memory, and recovery_outcome (masked to attempted rows). Judge by **outcome_mcc** on the gold TEST split.

In [5]:
# 1. Clone the package + install deps, make it importable (re-run safe)
import os, sys
REPO = "https://github.com/Kiyas-Mahmud/webagent.git"
ROOT = "/kaggle/working/webagent"
SRC = f"{ROOT}/src"
if not os.path.isdir(ROOT):
    !git clone -b Code {REPO} {ROOT}
else:
    !cd {ROOT} && git pull --ff-only
%cd {ROOT}
!pip install -q -U "transformers>=4.49" peft bitsandbytes accelerate scikit-learn
for m in [k for k in list(sys.modules) if k == "web_agent" or k.startswith("web_agent.")]:
    del sys.modules[m]
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import web_agent
print("web_agent ready ->", list(web_agent.__path__))

Already up to date.
/kaggle/working/webagent
web_agent ready -> ['/kaggle/working/webagent/src/web_agent']


In [6]:
# 2. GPU + gold data path (auto-detected, slug-proof)
import os, glob
!nvidia-smi -L
print("input dirs:", os.listdir("/kaggle/input"))
hits = glob.glob("/kaggle/input/**/split_train.json", recursive=True)
assert hits, "WebGoldData not attached: right panel -> Add Input -> WebGoldData"
GOLD_PATH = os.path.dirname(hits[0])
print("GOLD_PATH =", GOLD_PATH)
print("splits:", [f for f in os.listdir(GOLD_PATH) if f.endswith(".json")])

/bin/bash: line 1: nvidia-smi: command not found
input dirs: ['datasets']
GOLD_PATH = /kaggle/input/datasets/kiyasmahmud/webgolddata/web_agent_gold_v8_approved_2032_real_browser
splits: ['split_test.json', 'gold_export_summary.json', 'split_train.json', 'split_val.json', 'blank_filter_summary.json']


In [ ]:
# 3. Config (gold) + processor + one gold batch
import torch
from transformers import AutoProcessor
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
from web_agent.data.gold_dataloader import build_gold_dataloader, load_gold_split

set_seed(42)
cfg = load_config("configs/backbones/qwen2vl_2b_gold.yaml")
cfg["data"]["root"] = GOLD_PATH
cfg["data"]["num_workers"] = 0

bb = cfg["backbone"]
processor = AutoProcessor.from_pretrained(
    bb["vlm_model"], min_pixels=bb["min_pixels"], max_pixels=bb["max_pixels"])

train = load_gold_split(cfg, "train")
print("gold train rows:", len(train))
loader = build_gold_dataloader(cfg, "train", processor, records=train,
                               limit=8, batch_size=4, num_workers=0)
batch = next(iter(loader))
print("batch keys:", list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"  {k:22} {tuple(v.shape)}  {v.dtype}")
print("outcome labels in batch:", batch["label_outcome"].tolist())

In [ ]:
# 4. Build model + gold class-weighted loss (v14: all 4 pillars ON)
from web_agent.models.model import WebAgentModel
from web_agent.models.loss import CombinedLoss
from web_agent.data.gold_dataloader import gold_class_weights
from web_agent.data.gold_dataset import view

model = WebAgentModel(cfg)
print("VLM hidden dim D =", model.encoder.hidden_dim, "| pooling =", cfg["backbone"]["pooling"])
device = "cuda"
for m in (model.adapter, model.failure_head, model.action_head,
          model.memory_head, model.recovery_outcome_head):
    m.to(device)

aw, fw, ow = gold_class_weights(train)
# recovery_success pos_weight from the ATTEMPTED rows only (True/False; nulls ignored).
rs_vals = [view(r)[1].get("recovery_success") for r in train]
pos = sum(x is True for x in rs_vals); neg = sum(x is False for x in rs_vals)
rsw = torch.tensor([min(neg / max(pos, 1), 5.0)])
print("action w :", [round(x,2) for x in aw.tolist()], "(6 classes incl PRESS_KEY)")
print("failtype w:", [round(x,2) for x in fw.tolist()])
print("outcome w:", [round(x,2) for x in ow.tolist()], "(capped)")
print(f"recovery pos_weight: {float(rsw):.2f}  (attempted: True {pos} / False {neg})")
loss_fn = CombinedLoss(cfg, action_class_weights=aw.to(device),
                       failtype_class_weights=fw.to(device),
                       outcome_class_weights=ow.to(device),
                       recovery_success_pos_weight=rsw.to(device)).to(device)
print("head loss weights -> confidence:", cfg["loss"]["confidence"],
      "| memory:", cfg["loss"]["memory_flag"],
      "| recovery_outcome:", cfg["loss"]["recovery_outcome"], "(all 4 pillars ON)")
print("trainable params:", f"{sum(p.numel() for p in model.trainable_parameters()):,}")

In [ ]:
# 5. SMOKE — one batch, loss finite, all 4 pillars training
model.train()
b = next(iter(loader))
with torch.autocast("cuda", dtype=torch.float16):
    preds = model(b)
    terms = loss_fn(preds, {k: (v.to(device) if torch.is_tensor(v) else v)
                            for k, v in b.items()})
print("loss terms:", {k: round(float(v.detach()), 4) for k, v in terms.items()})
assert torch.isfinite(terms["total"]), "loss NaN/inf - STOP"
# v14: confidence + memory + recovery_outcome all trained. (recovery_outcome may be 0 in a
# batch with no attempted-recovery rows -- that's fine, it fires when such rows appear.)
print("head terms -> confidence:", round(float(terms["confidence"].detach()), 4),
      "| memory:", round(float(terms["memory"].detach()), 4),
      "| recovery_outcome:", round(float(terms["recovery_outcome"].detach()), 4))
print("contrastive non-zero:", float(terms["contrastive"].detach()) != 0.0)
print("peak GPU (GB):", round(torch.cuda.max_memory_allocated()/1e9, 2))
print("SMOKE PASS")

In [ ]:
# 6. TRAIN on gold + final eval on the gold TEST split (all 4 pillars)
import numpy as np
from web_agent.train.trainer import Trainer, collect_predictions, compute_metrics
from web_agent.eval.metrics import accuracy, outcome_mcc

cfg["train"]["epochs"] = 5
cfg["data"]["num_workers"] = 4
train_loader = build_gold_dataloader(cfg, "train", processor, shuffle=True, num_workers=4)
val_loader   = build_gold_dataloader(cfg, "val",   processor, shuffle=False, num_workers=4)

trainer = Trainer(model, loss_fn, cfg, train_loader, val_loader, train_sampler=None)
print("training done:", trainer.fit())

test_loader = build_gold_dataloader(cfg, "test", processor, shuffle=False, num_workers=4)
p = collect_predictions(model, test_loader, device)
gold_metrics = compute_metrics(p)

# recovery_outcome: score ONLY the attempted rows (label != -1). Overwrite the raw
# (null-contaminated) recovery_outcome_acc and add a real MCC.
rot = np.asarray(p["recovery_outcome_true"]); rop = np.asarray(p["recovery_outcome_pred"])
msk = rot >= 0
gold_metrics["recovery_outcome_acc"] = accuracy(rot[msk], rop[msk]) if msk.any() else 0.0
gold_metrics["recovery_outcome_mcc"] = outcome_mcc(rot[msk], rop[msk]) if msk.any() else 0.0

print("GOLD TEST:", {k: round(v, 4) for k, v in gold_metrics.items()})
print("  HEADLINE = outcome_mcc / outcome_bal_acc / failure_macro_f1")
print(f"  ALL 4 PILLARS real. recovery_outcome scored on {int(msk.sum())} attempted rows.")

In [ ]:
# 7. Save the gold-TEST result to ONE CSV -> download from the Output panel
import csv as _csv
assert "gold_metrics" in globals(), "Run cell 6 to completion first (it defines gold_metrics)."
VERSION = "gold_v14"   # <-- change per run (gold_v14, gold_v15, ...)

csv_path = f"/kaggle/working/gold_test_{VERSION}.csv"
with open(csv_path, "w", newline="") as f:
    w = _csv.writer(f)
    w.writerow(["version"] + list(gold_metrics.keys()))
    w.writerow([VERSION] + [round(v, 5) for v in gold_metrics.values()])

print("SAVED CSV:", csv_path)
print("GOLD TEST:", {k: round(v, 4) for k, v in gold_metrics.items()})
print("Download it from the right-side Output panel (/kaggle/working).")